In [ ]:
import json
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from matplotlib import animation
from IPython.display import HTML, Video, display
import tqdm as tqdm
from skimage.feature import local_binary_pattern
from scipy.spatial.distance import cdist
from scipy.spatial.distance import pdist
import os

In [ ]:
df = pd.read_csv("/home/guilherme_monteiro/projetos/tcc/data/metadata/video-metadata-publish-with-links.csv")
df['video_path'] = df['Filename'].apply(lambda x: "/home/guilherme_monteiro/projetos/tcc/data/videos/" + x)
df_true = df[df['Video Ground Truth'] == "Real"].reset_index(drop=True)
df_fake = df[df['Video Ground Truth'] == "Fake"].reset_index(drop=True)

# Funções básicas

In [ ]:
def load_video_frames(video_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)

    frames = []
    count = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frames.append(frame)

        count += 1
        if max_frames and count >= max_frames:
            break

    cap.release()

    return np.array(frames)

def standardize_frame(frame, max_size=640):
    h, w = frame.shape[:2]

    scale = min(max_size / max(h, w), 1.0)
    new_w, new_h = int(w * scale), int(h * scale)

    frame_resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

    return frame_resized, scale

def load_metadata(video_path):
    path = video_path.replace(".mp4", "_meta.json")

    full_path = os.path.join("/home/guilherme_monteiro/projetos/tcc/data/metadata", os.path.basename(path))

    with open(full_path, "r") as f:
        return json.load(f)

def create_face_regions(frame, bbox, padding=0.2):
    h, w = frame.shape[:2]

    x1, y1, x2, y2 = bbox

    bw = x2 - x1
    bh = y2 - y1

    # aplica padding
    px = int(bw * padding)
    py = int(bh * padding)

    x1p = max(0, x1 - px)
    y1p = max(0, y1 - py)
    x2p = min(w, x2 + px)
    y2p = min(h, y2 + py)

    # máscara face interna
    face_mask = np.zeros((h, w), dtype=np.uint8)
    face_mask[y1:y2, x1:x2] = 1

    # máscara expandida
    expanded_mask = np.zeros((h, w), dtype=np.uint8)
    expanded_mask[y1p:y2p, x1p:x2p] = 1

    # borda = expandida - interna
    border_mask = expanded_mask - face_mask

    # fundo = resto
    background_mask = 1 - expanded_mask

    return {
        "face": face_mask,
        "border": border_mask,
        "background": background_mask,
        "bbox_expanded": (x1p, y1p, x2p, y2p)
    }


## Funções pra visualização

In [ ]:
def draw_text_block(img, text_lines, x=10, y=20, line_height=18):
    for i, line in enumerate(text_lines):
        cv2.putText(
            img,
            line,
            (x, y + i * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
    return img

# Desenha a caixa delimitadora no frame
def draw_face_box(frame, bbox, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = bbox
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def overlay_regions(frame, regions):
    overlay = frame.copy()

    face_mask = regions["face"]
    border_mask = regions["border"]
    background_mask = regions["background"]

    overlay[face_mask == 1] = (0, 255, 0)       # verde
    overlay[border_mask == 1] = (0, 0, 255)     # vermelho
    overlay[background_mask == 1] = (255, 0, 0) # azul

    alpha = 0.3
    return cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

# FFT

In [ ]:
def compute_fft(gray):

    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)

    magnitude = np.abs(fshift)

    return magnitude

In [ ]:
# mascara especial para FFT, para isolar frequências específicas
def create_fft_masks(shape):

    h, w = shape
    cy, cx = h // 2, w // 2

    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2)

    # baixa frequência (centro)
    radius = min(h, w) * 0.1
    center_mask = dist <= radius

    # cruz (eixos)
    axis_thickness = 3
    cross_mask = (
        (np.abs(X - cx) < axis_thickness) |
        (np.abs(Y - cy) < axis_thickness)
    )

    return center_mask, cross_mask

In [ ]:
def compute_fft_region(mag, mask):

    if np.sum(mask) < 50:
        return None

    h, w = mag.shape
    cy, cx = h // 2, w // 2

    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2)
    max_r = dist.max()

    region_mag = mag * mask
    valid_mask = (mask == 1)

    total_energy = np.sum(region_mag) + 1e-6

    # dc, mede a energia na região central (baixa frequência)
    dc_mask = dist <= (0.1 * max_r)
    dc_energy = np.sum(region_mag[dc_mask]) / total_energy

    # alta frequencia, mede a energia nas bordas (frequências altas)
    high_mask = dist > (0.6 * max_r)
    high_freq = np.sum(region_mag[high_mask]) / total_energy

    # entropia, mede a distribuição de energia (complexidade)
    values = region_mag[valid_mask]
    values = values[values > 1e-8]
    values = values / (np.sum(values) + 1e-6)

    entropy = -np.sum(values * np.log(values + 1e-6))

    # anistropia, mede a direção predominante da energia
    angle = np.arctan2(Y - cy, X - cx)

    bins = np.linspace(-np.pi, np.pi, 9)
    energy_bins = []

    for i in range(len(bins) - 1):
        ang_mask = (angle >= bins[i]) & (angle < bins[i+1])
        energy_bins.append(np.sum(region_mag[ang_mask & valid_mask]))

    energy_bins = np.array(energy_bins)
    anisotropy = np.std(energy_bins) / (np.mean(energy_bins) + 1e-6)

    # simetria, mede a diferença entre a região e suas versões espelhadas
    flip_h = np.flipud(region_mag)
    flip_v = np.fliplr(region_mag)

    sym_h = np.mean(np.abs(region_mag - flip_h)[valid_mask])
    sym_v = np.mean(np.abs(region_mag - flip_v)[valid_mask])

    return {
        "dc_ratio": dc_energy,
        "high_freq_ratio": high_freq,
        "entropy": entropy,
        "anisotropy": anisotropy,
        "symmetry_h": sym_h,
        "symmetry_v": sym_v
    }

def fft_region_differences(f1, f2, prefix): 
    if f1 is None or f2 is None: 
        return {} 
    
    return { 
        f"{prefix}_fft_dc_diff": abs(f1["dc_ratio"] - f2["dc_ratio"]), 
        f"{prefix}_fft_high_freq_diff": abs(f1["high_freq_ratio"] - f2["high_freq_ratio"]), 
        f"{prefix}_fft_entropy_diff": abs(f1["entropy"] - f2["entropy"]), 
        f"{prefix}_fft_anisotropy_diff": abs(f1["anisotropy"] - f2["anisotropy"]), 
        }

In [ ]:
def compute_fft_metrics(frame, bbox):

    frame_std, _ = standardize_frame(frame)
    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

    mag = compute_fft(gray)

    regions = create_face_regions(frame_std, bbox)

    face = compute_fft_region(mag, regions["face"])
    border = compute_fft_region(mag, regions["border"])
    bg = compute_fft_region(mag, regions["background"])

    features = {}

    # diretas
    if face:
        features["fft_face_dc"] = face["dc_ratio"]
        features["fft_face_high_freq"] = face["high_freq_ratio"]
        features["fft_face_entropy"] = face["entropy"]
        features["fft_face_anisotropy"] = face["anisotropy"]

    # derivadas (ESSENCIAIS)
    features.update(fft_region_differences(face, bg, "face_bg"))
    features.update(fft_region_differences(face, border, "face_border"))
    features.update(fft_region_differences(border, bg, "border_bg"))

    return features, mag

In [ ]:
def fft_visual(mag):

    mag_log = np.log(mag + 1)

    mag_vis = cv2.normalize(mag_log, None, 0, 255, cv2.NORM_MINMAX)

    return mag_vis.astype(np.uint8)

def debug_fft_video(video_path, max_frames=500, interval=60):

    frames = load_video_frames(video_path)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    metadata = load_metadata(video_path)

    debug_frames = []

    print("\nProcessando FFT...")

    for i, frame in enumerate(tqdm.tqdm(frames)):

        frame_std, scale = standardize_frame(frame)
        gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

        bbox = metadata[i]["bbox"]
        bbox_scaled = [int(x * scale) for x in bbox]

        regions = create_face_regions(frame_std, bbox_scaled)

        features, mag = compute_fft_metrics(frame, bbox_scaled)

        fft_vis = fft_visual(mag)

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {i}",
            f"Face DC: {features.get('fft_face_dc', 0):.3f}",
            f"Face High Freq: {features.get('fft_face_high_freq', 0):.3f}",
            f"Face Entropy: {features.get('fft_face_entropy', 0):.3f}",
            f"Face Aniso: {features.get('fft_face_anisotropy', 0):.3f}",
            f"Face-BG DC Diff: {features.get('face_bg_fft_dc_diff', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(fft_vis, cv2.COLOR_GRAY2RGB)
        ])

        debug_frames.append(combined)

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")

    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(debug_frames),
        interval=interval,
        blit=True,
    )

    plt.close(fig)
    display(HTML(ani.to_jshtml()))

In [ ]:
debug_fft_video(df_fake['video_path'][1], max_frames=200)

# Todas as metricas

In [ ]:
def all_metrics(video_path, max_frames=500, label=None):
    video_name = os.path.basename(video_path)
    print(f"\nExtraindo métricas do vídeo: {video_name}")

    frames = load_video_frames(video_path)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    metadata = load_metadata(video_path)

    all_features = []

    for i, frame in enumerate(frames):
        frame_std, scale = standardize_frame(frame)

        bbox = metadata[i]["bbox"]
        bbox_scaled = [int(x * scale) for x in bbox]

        features_noise, _ = compute_fft_metrics(frame_std, bbox_scaled)

        features = {**features_noise}
        features["frame"] = i

        all_features.append(features)

    df = pd.DataFrame(all_features)
    df.insert(0, "video_name", video_name)
    if label is not None:
        df.insert(1, "label", label)

    return df